## Experimento 2:
1. Diminuição do Learning Rate de 0.001 para 0.0001
2. Obs.: Observar Loss do ResNet18 -> curva mais estável, treino consistente e talvez melhor resultado

In [1]:
# Importação das bibliotecas e inicialização dos dados
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, classification_report
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) # Onde o programa vai ser executado, se existir uma gpu compatível com cuda (nvidia) disponível, usa a GPU (mais rapido), se não vai para cpu

train_path = "dataset_processed/train"
test_path  = "dataset_processed/test"

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

cpu


In [2]:
# Dados de Treino
train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
test_dataset  = datasets.ImageFolder(test_path, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

classes = train_dataset.classes
print(classes)

['asphalt', 'belgian_blocks', 'offroad']


In [3]:
# Função Treino
def train_model(model, epochs=10):

    model.to(device)

    criterion = nn.CrossEntropyLoss() # mede quão longe a previsão do modelo está da resposta correta (padrão para multiclasse)
    optimizer = optim.Adam(model.parameters(), lr=0.0001) # MUDANÇA

    start = time.time()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad() # zera gradientes antigos
            outputs = model(images) # as imagens entram no modelo
            loss = criterion(outputs, labels) # compara as saidas com o gabarito e calcula a loss
            loss.backward() # calcuça quanto cada peso contribuiu para o erro
            optimizer.step() # atualiza os pesos

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss:.4f}")

    end = time.time()

    print("Tempo treino:", round(end-start,2), "segundos")

In [4]:
# Função Teste
def evaluate_model(model):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images) # gera as previsões
            _, preds = torch.max(outputs, 1) # escolhe a classe com o maior valor

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred) # calcula a porcentaggem de acertos

    print("Accuracy:", accuracy)
    print(classification_report(y_true, y_pred, target_names=classes)) # mostra as métricas por classe

In [5]:
# Avaliação do ResNet18
print("\n##### RESNET18 #####")

resnet = models.resnet18(weights="DEFAULT") # carrega o modelo
resnet.fc = nn.Linear(resnet.fc.in_features, 3) # muda para 3 classes

train_model(resnet, epochs=10) # treina
evaluate_model(resnet) # testa

# Avaliação do EfficientNet-B0
print("\n##### EFFICIENTNET_B0 #####")

effnet = models.efficientnet_b0(weights="DEFAULT")
effnet.classifier[1] = nn.Linear(
    effnet.classifier[1].in_features, 3
)

train_model(effnet, epochs=10)
evaluate_model(effnet)


##### RESNET18 #####
Epoch 1/10 Loss: 8.9410
Epoch 2/10 Loss: 2.4079
Epoch 3/10 Loss: 0.8817
Epoch 4/10 Loss: 0.5639
Epoch 5/10 Loss: 0.5262
Epoch 6/10 Loss: 0.4362
Epoch 7/10 Loss: 0.2328
Epoch 8/10 Loss: 0.1315
Epoch 9/10 Loss: 0.1219
Epoch 10/10 Loss: 0.2972
Tempo treino: 476.72 segundos
Accuracy: 0.93
                precision    recall  f1-score   support

       asphalt       0.93      1.00      0.96       218
belgian_blocks       0.94      0.50      0.65        32
       offroad       0.92      0.92      0.92        50

      accuracy                           0.93       300
     macro avg       0.93      0.81      0.85       300
  weighted avg       0.93      0.93      0.92       300


##### EFFICIENTNET_B0 #####
Epoch 1/10 Loss: 19.3837
Epoch 2/10 Loss: 5.9650
Epoch 3/10 Loss: 2.9807
Epoch 4/10 Loss: 1.5828
Epoch 5/10 Loss: 0.7856
Epoch 6/10 Loss: 0.4825
Epoch 7/10 Loss: 0.6256
Epoch 8/10 Loss: 0.5647
Epoch 9/10 Loss: 0.3308
Epoch 10/10 Loss: 0.5265
Tempo treino: 508.92 segun

## Analise dos Resultados:
### Experimento 2 – Redução da Learning Rate

Mudança:
```
0.001 → 0.0001
```
Objetivo:
Reduzir instabilidade observada na loss

### ResNet18 + LR 0.0001:

| Métrica  | Baseline | Experimento 2 | Variação |
| -------- | -------- | ------------- | -------- |
| Accuracy | 0.8167   | 0.9300        | +0.1133  |
| Macro F1 | 0.61     | 0.85          | +0.24    |


### Mudanças por classe:

| Classe         | Recall Baseline | Recall Exp2 |
| -------------- | --------------- | ----------- |
| Belgian Blocks | 0.25            | 0.50        |
| Off-road       | 0.38            | 0.92        |

-> Foi o melhor resultado do ResNet18, a curva da loss ficou estável e houve um ganho forte nas classes minoritárias.


### EfficientNet-B0 + LR 0.0001

| Métrica  | Baseline | Experimento 2 | Variação |
| -------- | -------- | ------------- | -------- |
| Accuracy | 0.9033   | 0.9300        | +0.0267  |
| Macro F1 | 0.78     | 0.84          | +0.06    |

### Mudanças por classe

| Classe         | Recall Baseline | Recall Exp2 |
| -------------- | --------------- | ----------- |
| Belgian Blocks | 0.38            | 0.53        |

-> A redução da learning rate também beneficiou a EfficientNet-B0, principalmente nos testes da classe belgian_blocks